In [16]:
import os
from pathlib import Path
import pdfplumber
CACHE_FOLDER="/home/jovyan/llmgenai/hf_cache"

In [1]:
import hashlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pdfplumber


# --- Worker functions placed at module-level for clean multiprocessing pickling ---

def _calculate_md5(file_path: Path) -> str:
    """Calculate the MD5 hash of a file by reading it in binary chunks."""
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()


def _read_existing_md5(txt_path: Path) -> str | None:
    """Read the first line of an existing text file to extract the stored MD5 hash."""
    if not txt_path.exists():
        return None
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            first_line = f.readline().strip()
            if first_line.startswith("# MD5:"):
                return first_line.split("# MD5:")[1].strip()
    except Exception:
        return None
    return None


def _is_char_in_bbox(char: dict, bbox: tuple) -> bool:
    """Check if a character's coordinates fall inside a table bounding box."""
    c_x0 = char.get("x0", 0)
    c_top = char.get("top", 0)
    c_x1 = char.get("x1", 0)
    c_bottom = char.get("bottom", 0)

    t_x0, t_top, t_x1, t_bottom = bbox
    return not (c_x1 < t_x0 or c_x0 > t_x1 or c_bottom < t_top or c_top > t_bottom)


def _format_table_as_markdown(table: list) -> str:
    """Convert a 2D list (table) into a Markdown grid string."""
    if not table or not any(table):
        return ""

    clean_table = [
        [str(cell).replace("\n", " ").strip() if cell else "" for cell in row]
        for row in table
    ]
    cols = max(len(row) for row in clean_table)
    clean_table = [row + [""] * (cols - len(row)) for row in clean_table]

    headers = clean_table[0]
    rows = clean_table[1:]

    md_lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * cols) + " |",
    ]
    for row in rows:
        md_lines.append("| " + " | ".join(row) + " |")

    return "\n".join(md_lines)


import re

def strip_metadata_headers(text: str) -> str:
    """Removes lines starting with '# MD5:' or similar comment markers."""
    cleaned_text = re.sub(r"^#\s*MD5:\s*[a-fA-F0-9]{32}\s*\n?", "", text, flags=re.MULTILINE)
    return cleaned_text.strip()


def _process_single_pdf_worker(args: tuple) -> tuple[str, dict]:
    pdf_path, output_filepath, x_tolerance, y_tolerance = args
    pdf_path = Path(pdf_path)
    output_filepath = Path(output_filepath)

    current_md5 = _calculate_md5(pdf_path)

    # Check if output exists and matches hash
    existing_md5 = _read_existing_md5(output_filepath)
    if existing_md5 == current_md5:
        print(f"[SKIP] Unmodified: {pdf_path.name}")
        
        # Read disk content and clean header via regex function
        raw_disk_text = output_filepath.read_text(encoding="utf-8")
        clean_text = strip_metadata_headers(raw_disk_text)
        
        return str(pdf_path), {
            "extracted_text": clean_text,
            "output_filename": str(output_filepath),
            "md5": current_md5,
            "status": "skipped_unmodified",
        }

    # Process new or modified file
    print(f"[PROCESS] ({'modified' if existing_md5 else 'new'}): {pdf_path.name}")
    page_text_blocks = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.find_tables()
            table_bboxes = [t.bbox for t in tables]
            extracted_tables = page.extract_tables()

            def filter_out_tables(obj):
                if obj.get("object_type") == "char":
                    return not any(
                        _is_char_in_bbox(obj, bbox) for bbox in table_bboxes
                    )
                return True

            filtered_page = page.filter(filter_out_tables)
            raw_text = (
                filtered_page.extract_text(
                    x_tolerance=x_tolerance, y_tolerance=y_tolerance
                )
                or ""
            )
            formatted_md_tables = [
                _format_table_as_markdown(t) for t in extracted_tables
            ]

            page_content = []
            if raw_text.strip():
                page_content.append(raw_text.strip())

            if formatted_md_tables:
                page_content.append("\n[EXTRACTED TABLES]:\n")
                page_content.extend(formatted_md_tables)

            if page_content:
                page_text_blocks.append("\n\n".join(page_content))

    clean_extracted_text = "\n\n".join(page_text_blocks).strip()

    # Save to disk with MD5 header for caching
    disk_file_content = f"# MD5: {current_md5}\n\n{clean_extracted_text}"
    output_filepath.parent.mkdir(parents=True, exist_ok=True)
    output_filepath.write_text(disk_file_content, encoding="utf-8")

    return str(pdf_path), {
        "extracted_text": clean_extracted_text,
        "output_filename": str(output_filepath),
        "md5": current_md5,
        "status": "processed",
    }


class PdfToTxt:
    """
    A multiprocess-enabled class to recursively process directory structures of PDFs.
    Filters raw unformatted tables and inserts clean Markdown tables.
    """

    def __init__(
        self,
        x_tolerance: float = 2.0,
        y_tolerance: float = 3.0,
        max_workers: int | None = None,
    ):
        self.x_tolerance = x_tolerance
        self.y_tolerance = y_tolerance
        # Default to max available CPU cores if not specified
        self.max_workers = max_workers or os.cpu_count() or 4

    def process_pdf_directory_deep(self, input_dir: str, output_dir: str) -> dict:
        """
        Recursively discovers all PDFs and processes them concurrently across CPU workers.
        """
        input_base = Path(input_dir).resolve()
        output_base = Path(output_dir).resolve()

        # Build execution tasks list
        tasks = []
        for pdf_path in input_base.rglob("*.pdf"):
            relative_path = pdf_path.relative_to(input_base)
            output_filepath = (output_base / relative_path).with_suffix(".txt")

            tasks.append((
                str(pdf_path),
                str(output_filepath),
                self.x_tolerance,
                self.y_tolerance,
            ))

        pdf_dict = {}
        if not tasks:
            print("No PDF files found.")
            return pdf_dict

        print(f"Starting multiprocessing across {self.max_workers} worker processes for {len(tasks)} files...")

        # Process tasks in parallel using ProcessPoolExecutor
        with ProcessPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [
                executor.submit(_process_single_pdf_worker, task) for task in tasks
            ]

            for future in as_completed(futures):
                try:
                    pdf_key, result_data = future.result()
                    pdf_dict[pdf_key] = result_data
                except Exception as e:
                    print(f"Worker process failed with error: {e}")

        return pdf_dict

def load_pdf(INPUT_FOLDER, OUTPUT_FOLDER):
    # Instantiate the class
    converter = PdfToTxt(x_tolerance=1.5, y_tolerance=3, max_workers=6)
    
    # Process all PDFs recursively
    results = converter.process_pdf_directory_deep(INPUT_FOLDER, OUTPUT_FOLDER)
    
    print(f"\nTotal files processed: {len(results)}")
    return results
    

In [2]:

INPUT_FOLDER = "data/cisco-dc"
OUTPUT_FOLDER = "data/text/cisco-dc"
# Load PDF and convert to text
pdf_dict = load_pdf(INPUT_FOLDER,OUTPUT_FOLDER)


Starting multiprocessing across 6 worker processes for 65 files...
[SKIP] Unmodified: 02_compare_features_select_software_release.pdf[SKIP] Unmodified: 01_bfd_for_bgp_admindown_nexus7000.pdf

[SKIP] Unmodified: 03_create_giso_iosxr_python3.pdf
[SKIP] Unmodified: 01_nexus9000_nxos_fundamentals_10.5x.pdf[SKIP] Unmodified: 04_nexus9000_nxos_unicast_routing_10.5x.pdf

[SKIP] Unmodified: 02_nexus9000_nxos_interfaces_10.5x.pdf
[SKIP] Unmodified: 05_exclude_oid_nexus_snmp.pdf
[SKIP] Unmodified: 04_nx_sdk_python_nexus3000_9000.pdf
[SKIP] Unmodified: 06_fex_power_supply_failure_troubleshooting.pdf
[SKIP] Unmodified: 07_nexus_nxos_tips_tricks.pdf[SKIP] Unmodified: 07_n9500_series_data_sheet.pdf

[SKIP] Unmodified: 06_n9300_platform_data_sheet.pdf
[SKIP] Unmodified: 08_n9300_aci_fixed_spine_data_sheet.pdf
[SKIP] Unmodified: 08_otv_site_vlans_aed_election.pdf[SKIP] Unmodified: 09_apic_data_sheet.pdf[SKIP] Unmodified: 09_tac_requested_outputs_nexus.pdf
[SKIP] Unmodified: 10_nexus_dashboard_data_she

In [3]:
import hashlib
import json
import re
import unicodedata
from collections import Counter
from datetime import datetime
from pathlib import Path

from datasketch import MinHash, MinHashLSH
from langdetect import DetectorFactory, LangDetectException, detect


# langdetect is non-deterministic unless its random seed is fixed. Reproducible
# classifications are important for a cleaning audit.
DetectorFactory.seed = 0


_EXTRACTED_TABLES_MARKER = re.compile(
    r"^[ \t]*\[EXTRACTED TABLES\]:?[ \t]*$", flags=re.IGNORECASE
)


def _normalize_edge_line(line: str) -> str:
    """Canonicalize a line only for repeated header/footer comparison."""
    return re.sub(r"\s+", " ", line).strip().casefold()


def _is_header_footer_candidate(line: str) -> bool:
    """Conservatively identify prose-like page-edge boilerplate."""
    normalized = _normalize_edge_line(line)
    if not 12 <= len(normalized) <= 200:
        return False

    # Preserve Markdown tables, fenced code, and common CLI/configuration lines.
    if line.lstrip().startswith(("|", "```")):
        return False
    if any(token in line for token in ("#", "{", "}")):
        return False

    words = re.findall(r"[A-Za-z][A-Za-z0-9-]*", normalized)
    return len(words) >= 4


def _is_page_like_block(block: str) -> bool:
    """Return True for an extracted page-text block, excluding table blocks."""
    lines = [line for line in block.splitlines() if line.strip()]
    if len(lines) == 1 and _EXTRACTED_TABLES_MARKER.fullmatch(lines[0]):
        return False
    if len(lines) < 4:
        return False

    markdown_table_lines = sum(line.lstrip().startswith("|") for line in lines)
    return markdown_table_lines <= len(lines) / 2


def _edge_line_indices(lines: list[str], edge_size: int = 4) -> set[int]:
    """Return the first and last non-empty line positions in a page block."""
    non_empty = [index for index, line in enumerate(lines) if line.strip()]
    return set(non_empty[:edge_size] + non_empty[-edge_size:])


def _remove_repeated_page_headers_footers(text: str) -> str:
    """Remove lines repeatedly found at page-like block edges.

    PDF extraction generally keeps a page's prose as one multiline block and
    separates pages, table markers, and Markdown tables with blank lines. This
    lets us identify repeated page-edge text without deleting repeated commands
    or table rows from the document body.
    """
    blocks = re.split(r"\n{2,}", text)
    page_block_indices = [
        index for index, block in enumerate(blocks) if _is_page_like_block(block)
    ]
    if len(page_block_indices) < 3:
        return text

    edge_counts: Counter[str] = Counter()
    for block_index in page_block_indices:
        lines = blocks[block_index].splitlines()
        candidates = {
            _normalize_edge_line(lines[line_index])
            for line_index in _edge_line_indices(lines)
            if _is_header_footer_candidate(lines[line_index])
        }
        edge_counts.update(candidates)

    # A line must appear at the edge of at least three page-like blocks and on
    # at least 5% of them. This avoids treating occasional repeated headings as
    # boilerplate in long documents.
    minimum_occurrences = max(3, (len(page_block_indices) + 19) // 20)
    repeated_edge_lines = {
        line for line, count in edge_counts.items() if count >= minimum_occurrences
    }
    if not repeated_edge_lines:
        return text

    for block_index in page_block_indices:
        lines = blocks[block_index].splitlines()
        edge_indices = _edge_line_indices(lines)
        blocks[block_index] = "\n".join(
            line
            for line_index, line in enumerate(lines)
            if not (
                line_index in edge_indices
                and _normalize_edge_line(line) in repeated_edge_lines
            )
        ).strip()

    return "\n\n".join(block for block in blocks if block.strip())


def strip_metadata_headers(text: str) -> str:
    """Produce the single canonical clean text used for training and gates.

    Removes extraction metadata and boilerplate while preserving document prose,
    CLI examples, and the contents and formatting of extracted Markdown tables.
    """
    # Normalize Windows and legacy Mac line endings first.
    cleaned_text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove the extraction cache header.
    cleaned_text = re.sub(
        r"^#\s*MD5:\s*[a-fA-F0-9]{32}\s*\n?",
        "",
        cleaned_text,
        flags=re.MULTILINE,
    )

    # Detect page-edge boilerplate before collapsing blank-line structure.
    cleaned_text = _remove_repeated_page_headers_footers(cleaned_text)

    # Remove only the table extraction label; actual Markdown tables remain.
    cleaned_text = re.sub(
        r"^[ \t]*\[EXTRACTED TABLES\]:?[ \t]*\n?",
        "",
        cleaned_text,
        flags=re.MULTILINE | re.IGNORECASE,
    )

    # Preserve tabs and newlines used by CLI examples and Markdown tables while
    # removing ASCII/Unicode control and formatting characters.
    cleaned_text = "".join(
        char
        for char in cleaned_text
        if char in ("\n", "\t") or unicodedata.category(char) not in {"Cc", "Cf"}
    )

    # Remove trailing horizontal whitespace and cap blank runs at one blank line.
    cleaned_text = re.sub(r"[ \t]+\n", "\n", cleaned_text)
    cleaned_text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", cleaned_text)
    return cleaned_text.strip()


class TextCleaner:
    """
    Applies a 4-step quality filter pipeline to extracted document texts:
    1. Length filter (< min_chars)
    2. Paragraph repetition filter (> max_dup_paragraph_ratio)
    3. Flexible deduplication filter (exact MD5, near-duplicate MinHash/LSH, or both)
    4. Language filter (target language matching)

    Saves a detailed JSON audit report of all rejected files with lineage tracking.
    """

    def __init__(
        self,
        min_chars: int = 50,
        max_dup_paragraph_ratio: float = 0.30,
        target_lang: str = "en",
        dedup_strategy: str = "both",  # Options: "exact", "near", "both"
        similarity_threshold: float = 0.85,
        num_perm: int = 128,
    ):
        self.min_chars = min_chars
        self.max_dup_paragraph_ratio = max_dup_paragraph_ratio
        self.target_lang = target_lang
        self.dedup_strategy = dedup_strategy.lower()
        self.similarity_threshold = similarity_threshold
        self.num_perm = num_perm

        if self.dedup_strategy not in ("exact", "near", "both"):
            raise ValueError(
                "dedup_strategy must be one of: 'exact', 'near', or 'both'"
            )

        self._reset_deduplication_state()

    def _reset_deduplication_state(self) -> None:
        """Start a fresh deduplication index for one corpus-cleaning run."""
        # Exact hash -> (retained source file, retained extracted-text file)
        self.seen_md5_hashes: dict[str, tuple[str, str | None]] = {}

        # Structures for MinHash/LSH near-deduplication
        self.lsh = MinHashLSH(
            threshold=self.similarity_threshold, num_perm=self.num_perm
        )
        self.minhashes: dict[str, MinHash] = {}
        self.file_output_map: dict[str, str | None] = {}

    @staticmethod
    def _document_reference(
        source_file: str, output_filename: str | None
    ) -> dict[str, str | None]:
        """Return the stable lineage fields used throughout the audit report."""
        return {
            "source_file": source_file,
            "output_filename": output_filename or None,
        }

    def _rejection_record(
        self,
        source_file: str,
        output_filename: str | None,
        reason: str,
        details: dict,
        retained_original: dict[str, str | None] | None = None,
    ) -> dict:
        """Build a consistent audit record for every rejected document."""
        return {
            **self._document_reference(source_file, output_filename),
            "status": "rejected",
            "reason": reason,
            # This is populated only for exact/near duplicates. Other rejection
            # reasons have no retained original, so the value is explicitly null.
            "retained_original": retained_original,
            "details": details,
        }

    def check_length(self, text: str) -> tuple[bool, int]:
        char_count = len(text.strip())
        return char_count >= self.min_chars, char_count

    def check_repetition(self, text: str) -> tuple[bool, float, int]:
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        if not paragraphs:
            return False, 1.0, 0

        # Ignore casing and PDF line-wrapping differences when comparing
        # paragraphs, otherwise repeated headers/footers can be missed.
        normalized_paragraphs = {
            re.sub(r"\s+", " ", paragraph).casefold()
            for paragraph in paragraphs
        }
        duplicate_count = len(paragraphs) - len(normalized_paragraphs)
        dup_ratio = duplicate_count / len(paragraphs)
        return dup_ratio <= self.max_dup_paragraph_ratio, dup_ratio, len(paragraphs)

    def check_exact_deduplication(
        self, text: str
    ) -> tuple[bool, str, tuple[str, str | None] | None]:
        doc_hash = hashlib.md5(text.encode("utf-8")).hexdigest()
        if doc_hash in self.seen_md5_hashes:
            return False, doc_hash, self.seen_md5_hashes[doc_hash]

        # Return the hash but DO NOT insert it into seen_md5_hashes yet
        return True, doc_hash, None

    def check_near_deduplication(
        self, text: str
    ) -> tuple[bool, float, tuple[str, str | None] | None, MinHash]:
        tokens = re.findall(r"\w+", text.lower())
        current_minhash = MinHash(num_perm=self.num_perm)

        if not tokens:
            return True, 0.0, None, current_minhash

        shingles = set(" ".join(tokens[i : i + 3]) for i in range(len(tokens) - 2))
        if not shingles:
            shingles = set(tokens)

        for shingle in shingles:
            current_minhash.update(shingle.encode("utf-8"))

        result_candidates = self.lsh.query(current_minhash)
        max_similarity = 0.0
        matching_file = None

        for candidate_path in result_candidates:
            candidate_minhash = self.minhashes[candidate_path]
            similarity = current_minhash.jaccard(candidate_minhash)
            if similarity > max_similarity:
                max_similarity = similarity
                matching_file = candidate_path

        if max_similarity >= self.similarity_threshold and matching_file:
            orig_output = self.file_output_map.get(matching_file)
            return (
                False,
                round(max_similarity, 4),
                (matching_file, orig_output),
                current_minhash,
            )

        # Return the minhash but DO NOT insert it into the LSH/maps yet
        return True, round(max_similarity, 4), None, current_minhash

    def check_deduplication(
        self, text: str
    ) -> tuple[bool, str, dict, str | None, MinHash | None]:
        """Routes text through deduplication and returns state to be saved if kept."""
        doc_hash = None
        current_minhash = None

        if self.dedup_strategy in ("exact", "both"):
            is_unique_exact, doc_hash, orig_info_exact = (
                self.check_exact_deduplication(text)
            )
            if not is_unique_exact:
                orig_file, orig_output = orig_info_exact
                return False, "failed_exact_deduplication", {
                    "doc_md5": doc_hash,
                    "retained_original": self._document_reference(
                        orig_file, orig_output
                    ),
                }, None, None

        if self.dedup_strategy in ("near", "both"):
            (
                is_unique_near,
                sim_score,
                orig_info_near,
                current_minhash,
            ) = self.check_near_deduplication(text)
            if not is_unique_near:
                orig_file, orig_output = orig_info_near
                return False, "failed_near_deduplication", {
                    "similarity_score": sim_score,
                    "threshold": self.similarity_threshold,
                    "retained_original": self._document_reference(
                        orig_file, orig_output
                    ),
                }, None, None

        return True, "passed", {}, doc_hash, current_minhash

    def check_language(self, text: str) -> tuple[bool, str]:
        try:
            detected_lang = detect(text[:2000])
            return detected_lang == self.target_lang, detected_lang
        except LangDetectException:
            return False, "unknown/unresolvable"

    def filter_corpus(
        self,
        pdf_dict: dict,
        audit_report_path: str | Path = "cleaning_audit_report.json",
    ) -> tuple[dict, dict]:
        # A report must be self-contained: duplicate references from this run
        # should never point to a retained document from an earlier invocation.
        self._reset_deduplication_state()

        clean_docs = {}
        rejected_docs = {}
        retained_document_records = []

        rejection_counts = {
            "failed_length_filter": 0,
            "failed_repetition_filter": 0,
            "failed_exact_deduplication": 0,
            "failed_near_deduplication": 0,
            "failed_language_filter": 0,
        }

        for file_path, data in pdf_dict.items():
            source_file = str(file_path)
            raw_text = data.get("extracted_text") or ""
            if not isinstance(raw_text, str):
                raw_text = str(raw_text)
            output_value = data.get("output_filename")
            output_filename = str(output_value) if output_value else None

            # Strip caching metadata headers (e.g., '# MD5: ...') before processing
            text = strip_metadata_headers(raw_text)

            # Step 1: Length Check
            is_valid_len, char_count = self.check_length(text)
            if not is_valid_len:
                reason_key = "failed_length_filter"
                rejection_counts[reason_key] += 1
                rejected_docs[file_path] = self._rejection_record(
                    source_file,
                    output_filename,
                    reason_key,
                    {
                        "char_count": char_count,
                        "min_chars_threshold": self.min_chars,
                    },
                )
                continue

            # Step 2: Repetition Check
            is_not_rep, dup_ratio, total_paras = self.check_repetition(text)
            if not is_not_rep:
                reason_key = "failed_repetition_filter"
                rejection_counts[reason_key] += 1
                rejected_docs[file_path] = self._rejection_record(
                    source_file,
                    output_filename,
                    reason_key,
                    {
                        "duplicate_paragraph_ratio": round(dup_ratio, 4),
                        "max_allowed_ratio": self.max_dup_paragraph_ratio,
                        "total_paragraphs": total_paras,
                    },
                )
                continue

            # Step 3: Deduplication Check
            (
                is_unique,
                reason_key,
                details,
                doc_hash,
                current_minhash,
            ) = self.check_deduplication(text)
            if not is_unique:
                rejection_counts[reason_key] += 1
                retained_original = details.pop("retained_original", None)
                rejected_docs[file_path] = self._rejection_record(
                    source_file,
                    output_filename,
                    reason_key,
                    details,
                    retained_original,
                )
                continue

            # Step 4: Language Check
            is_target_lang, detected_lang = self.check_language(text)
            if not is_target_lang:
                reason_key = "failed_language_filter"
                rejection_counts[reason_key] += 1
                rejected_docs[file_path] = self._rejection_record(
                    source_file,
                    output_filename,
                    reason_key,
                    {
                        "detected_language": detected_lang,
                        "expected_language": self.target_lang,
                    },
                )
                continue

            # Passed all filters - NOW it is safe to add to indexes
            if self.dedup_strategy in ("exact", "both") and doc_hash:
                self.seen_md5_hashes[doc_hash] = (source_file, output_filename)

            if self.dedup_strategy in ("near", "both") and current_minhash:
                self.lsh.insert(source_file, current_minhash)
                self.minhashes[source_file] = current_minhash
                self.file_output_map[source_file] = output_filename

            # Return the normalized text for downstream training while leaving
            # the caller's original input dictionary untouched for lineage.
            cleaned_data = dict(data)
            cleaned_data["extracted_text"] = text
            clean_docs[file_path] = cleaned_data
            retained_document_records.append(
                {
                    **self._document_reference(source_file, output_filename),
                    "status": "retained",
                    "details": {
                        "char_count": char_count,
                        "duplicate_paragraph_ratio": round(dup_ratio, 4),
                        "total_paragraphs": total_paras,
                        "detected_language": detected_lang,
                    },
                }
            )

        rejected_document_records = list(rejected_docs.values())

        # Guard against incomplete reports or dangling duplicate lineage.
        if len(clean_docs) + len(rejected_docs) != len(pdf_dict):
            raise RuntimeError("Cleaning audit does not account for every input file")
        if sum(rejection_counts.values()) != len(rejected_docs):
            raise RuntimeError("Cleaning audit rejection counts are inconsistent")

        retained_sources = {
            record["source_file"] for record in retained_document_records
        }
        for record in rejected_document_records:
            original = record["retained_original"]
            if original and original["source_file"] not in retained_sources:
                raise RuntimeError(
                    "Rejected duplicate references a document not retained "
                    "in this run: "
                    f"{original['source_file']}"
                )

        # Compile and export full JSON Audit Report
        audit_data = {
            "schema_version": 2,
            "timestamp": datetime.now().isoformat(),
            "summary": {
                "total_documents_processed": len(pdf_dict),
                "total_clean_kept": len(clean_docs),
                "total_rejected": len(rejected_docs),
                "rejection_breakdown": rejection_counts,
            },
            "configuration": {
                "min_chars": self.min_chars,
                "max_dup_paragraph_ratio": self.max_dup_paragraph_ratio,
                "target_lang": self.target_lang,
                "dedup_strategy": self.dedup_strategy,
                "similarity_threshold": self.similarity_threshold,
            },
            "retained_documents": retained_document_records,
            "rejected_documents": rejected_document_records,
        }

        report_file = Path(audit_report_path)
        report_file.parent.mkdir(parents=True, exist_ok=True)
        report_file.write_text(json.dumps(audit_data, indent=2), encoding="utf-8")

        print("\n--- Cleaning Summary ---")
        print(f"Strategy:        {self.dedup_strategy.upper()}")
        print(f"Total Processed: {len(pdf_dict)}")
        print(f"Clean Kept:      {len(clean_docs)}")
        print(f"Total Rejected:  {len(rejected_docs)}")
        print(f"Audit Log:       Saved to {report_file.resolve()}\n")

        return clean_docs, rejected_docs


def clean_data(
    data_dict,
    dedup="both",
    audit_report_path="cleaning_audit_report.json",
):
    cleaner = TextCleaner(
        min_chars=50,
        max_dup_paragraph_ratio=0.30,
        target_lang="en",
        dedup_strategy=dedup,
    )
    clean_results, rejected = cleaner.filter_corpus(
        data_dict, audit_report_path=audit_report_path
    )
    return clean_results, rejected


In [4]:
# clean the data
clean, rejected = clean_data(pdf_dict)


--- Cleaning Summary ---
Strategy:        BOTH
Total Processed: 65
Clean Kept:      61
Total Rejected:  4
Audit Log:       Saved to /home/jovyan/llmassign/1AE/cleaning_audit_report.json



In [7]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/home/jovyan/llmgenai/hf_cache"
from dotenv import load_dotenv
import numpy as np

load_dotenv()


from transformers import AutoTokenizer


def tokenize_gpt(clean_data):
    tokenizer = AutoTokenizer.from_pretrained(
        "gpt2",
        use_fast=True,
        cache_dir="/home/jovyan/llmgenai/hf_cache",
    )
    eos_id = tokenizer.eos_token_id
    context_length = tokenizer.model_max_length

    print(f"Tokenizer max length: {context_length}")
    paths = clean_data.keys()
    buffer = []
    all_packed_chunks = []
    doc_token_counts = []  # Track individual doc lengths for metrics
    for path in paths:
        print(
            "Tokenizing {}, text size {}".format(
                path, len(clean_data[path]["extracted_text"])
            )
        )
        token_ids = tokenizer.encode(
            clean_data[path]["extracted_text"],
            add_special_tokens=False,
            padding=False,
            truncation=False,
            return_attention_mask=False,
        )

        doc_token_counts.append(len(token_ids))
        token_ids.extend([eos_id])
        buffer.extend(token_ids)
        while len(buffer) >= context_length:
            chunk = buffer[:context_length]
            all_packed_chunks.append(chunk)
            buffer = buffer[context_length:]

    print("Tokenizing complete")

    # --- METRICS CALCULATIONS ---
    total_raw_tokens = sum(doc_token_counts)
    total_docs = len(doc_token_counts)
    avg_doc_length = total_raw_tokens / total_docs if total_docs > 0 else 0
    total_stream_tokens = total_raw_tokens + total_docs  # One EOS per document
    total_packed_seqs = len(all_packed_chunks)
    total_packed_tokens = total_packed_seqs * context_length

    # Print pipeline statistics
    print("==================================================")
    print("             CPT PIPELINE METRICS                 ")
    print("==================================================")
    print(f"Total Documents Processed:      {total_docs:,}")
    print(f"Document Tokens (without EOS):  {total_raw_tokens:,}")
    print(f"Total Token Count (with EOS):   {total_stream_tokens:,}")
    print(f"Average Document Length:        {avg_doc_length:.2f} tokens")
    print(f"Total Packed Sequences ({context_length}):  {total_packed_seqs:,}")
    print(f"Tokens in Packed Sequences:     {total_packed_tokens:,}")
    print(f"Residual Tokens Left in Buffer: {len(buffer):,}")
    print("==================================================\n")

    return all_packed_chunks, context_length


def _validate_packed_chunks(chunks):
    """Validate fixed-length sequence packing and return the context length."""
    if not chunks:
        raise ValueError("No packed sequences are available to save")

    context_length = len(chunks[0])
    if context_length == 0:
        raise ValueError("Packed sequences cannot be empty")

    invalid_indices = [
        index for index, chunk in enumerate(chunks) if len(chunk) != context_length
    ]
    if invalid_indices:
        preview = invalid_indices[:5]
        raise ValueError(
            "All packed sequences must have the same context length; "
            f"invalid sequence indices include {preview}"
        )

    return context_length


def save_chunks_to_parquet(
    chunks,
    filename,
    batch_size=1024,
    compression="zstd",
):
    """Save one fixed-length packed token sequence per Parquet row.

    The ``input_ids`` column is a fixed-size list, so each row represents one
    context-window-sized training example without padding. Batches are written
    incrementally to avoid constructing a second full in-memory copy.
    """
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError(
            "Parquet output requires pyarrow. Install it with: pip install pyarrow"
        ) from exc

    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")

    context_length = _validate_packed_chunks(chunks)
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    input_ids_type = pa.list_(pa.uint32(), context_length)
    schema = pa.schema(
        [
            ("sequence_id", pa.int64()),
            ("input_ids", input_ids_type),
        ],
        metadata={
            b"packing": b"concatenated_document_stream_no_padding",
            b"context_length": str(context_length).encode("utf-8"),
            b"total_packed_sequences": str(len(chunks)).encode("utf-8"),
            b"total_packed_tokens": str(
                len(chunks) * context_length
            ).encode("utf-8"),
        },
    )

    with pq.ParquetWriter(
        output_path,
        schema,
        compression=compression,
    ) as writer:
        for start in range(0, len(chunks), batch_size):
            batch = chunks[start : start + batch_size]
            token_matrix = np.asarray(batch, dtype=np.uint32)
            token_values = pa.array(
                token_matrix.reshape(-1),
                type=pa.uint32(),
            )
            input_ids = pa.FixedSizeListArray.from_arrays(
                token_values,
                context_length,
            )
            sequence_ids = pa.array(
                range(start, start + len(batch)),
                type=pa.int64(),
            )
            table = pa.Table.from_arrays(
                [sequence_ids, input_ids],
                schema=schema,
            )
            writer.write_table(table)

    print(
        f"Saved {len(chunks):,} packed sequences "
        f"({context_length} tokens each) to {output_path}"
    )


def save_chunks_to_disk(
    chunks,
    filename,
    parquet_filename=None,
    parquet_batch_size=1024,
):
    """Save the existing flat binary and optionally a Parquet copy.

    Existing calls with ``save_chunks_to_disk(chunks, filename)`` remain
    unchanged. Pass ``parquet_filename`` to write both formats in one call.
    """
    _validate_packed_chunks(chunks)

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    flat_tokens = np.asarray(chunks, dtype=np.uint16).reshape(-1)
    flat_tokens.tofile(output_path)
    print("Saved trainable binary chunks to", output_path)

    if parquet_filename is not None:
        save_chunks_to_parquet(
            chunks,
            parquet_filename,
            batch_size=parquet_batch_size,
        )


In [8]:

# convert to tokens
buffer, context_length = tokenize_gpt(clean)
save_chunks_to_disk(buffer, 'data/tokens.bin', parquet_filename='data/tokens.pqt', parquet_batch_size=context_length)


Token indices sequence length is longer than the specified maximum sequence length for this model (2411 > 1024). Running this sequence through the model will result in indexing errors


Tokenizer max length: 1024
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/03_create_giso_iosxr_python3.pdf, text size 7203
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/02_compare_features_select_software_release.pdf, text size 11310
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/01_bfd_for_bgp_admindown_nexus7000.pdf, text size 4936
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/01_nexus9000_nxos_fundamentals_10.5x.pdf, text size 295617
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/02_nexus9000_nxos_interfaces_10.5x.pdf, text size 865158
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/05_exclude_oid_nexus_snmp.pdf, text size 7541
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/04_nexus9000_nxos_unicast_routing_10.5x.pdf, text size 1273553
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/04_nx_sdk_python_nexus3000_9000.pdf, text size 48869
Tokenizing /home/jovyan/llmassign/1AE/data/cisco-dc/06_fex_power_supply_failure_troubleshooting.pdf, text size

In [14]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class MemmapDataset(Dataset):
    def __init__(self, bin_path: str, seq_len: int = 1024):
        self.seq_len = seq_len
        # Memory-map binary file (read-only)
        self.data = np.memmap(bin_path, dtype=np.uint16, mode="r")
        self.num_samples = len(self.data) // seq_len

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx: int):
        start = idx * self.seq_len
        end = start + self.seq_len
        
        # Pull slice from disk and convert directly to PyTorch LongTensor
        # Fast zero-copy tensor creation from numpy buffer + cast to long
        chunk = torch.from_numpy(self.data[start:end]).long()
        
        # Return input_ids and labels without unnecessary cloned allocations
        return {"input_ids": chunk, "labels": chunk}


def load_tokens_from_bin(
    filename: str, 
    context_length: int = 1024, 
    batch_size: int = 8, 
    num_workers: int = 4
) -> DataLoader:
    dataset = MemmapDataset(filename, seq_len=context_length)
    print(f"Total dataset samples: {len(dataset)}")
    print(f"Sample tensor shape:   {dataset[0]['input_ids'].shape}")  # torch.Size([1024])
    
    train_dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,        # Set to False if dataset is huge and disk I/O becomes bottleneck
        num_workers=num_workers, # Multi-process loading to eliminate GPU starvation
        pin_memory=True,     # Speeds up CPU -> GPU transfers
        drop_last=True,
    )
    
    return train_dataloader




    # Inspection loop


In [15]:
dataloader = load_tokens_from_bin("data/tokens.bin")
for step, batch in enumerate(dataloader):
    input_ids = batch["input_ids"] # [8, 1024]
    labels = batch["labels"]       # [8, 1024]
    
    print(f"Batch {step + 1} Shapes:")
    print(f"  input_ids: {input_ids.shape} | dtype: {input_ids.dtype}")
    print(f"  labels:    {labels.shape} | dtype: {labels.dtype}")
    
    break

Total dataset samples: 1615
Sample tensor shape:   torch.Size([1024])


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Batch 1 Shapes:
  input_ids: torch.Size([8, 1024]) | dtype: torch.int64
  labels:    torch.Size([8, 1024]) | dtype: torch.int64


In [34]:
def load_gpt2_model():
    from transformers import GPT2LMHeadModel, GPT2TokenizerFast
    
    # 1. Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # 2. Load pre-trained GPT-2 Tokenizer and Model
    model_name = "gpt2"  # Options: 'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'
    
    tokenizer = GPT2TokenizerFast.from_pretrained(model_name, cache_dir=CACHE_FOLDER)
    model = GPT2LMHeadModel.from_pretrained(model_name, cache_dir=CACHE_FOLDER)
    
    
    # 3. Move model to GPU/CPU
    model.to(device)
    
    # Enable gradient checkpointing to save VRAM if training on larger context sizes
    # model.gradient_checkpointing_enable()
    
    print(f"Successfully loaded {model_name} with {sum(p.numel() for p in model.parameters()):,} parameters.")
    return { 
        "model": model, 
        "tokenizer": tokenizer, 
        "device": device}


# Pull one batch from your previously defined train_dataloader
def check_model(dataloader, model, device):
   # Display the loaded model's architecture and parameter details

    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    
    print("\n" + "=" * 55)
    print("                 MODEL INFORMATION")
    print("=" * 55)
    print(f"Model name/path       : {model.config._name_or_path}")
    print(f"Model architecture    : {model.config.model_type}")
    print(f"Transformer layers    : {model.config.n_layer}")
    print(f"Attention heads       : {model.config.n_head}")
    print(f"Embedding dimension   : {model.config.n_embd}")
    print(f"Vocabulary size       : {model.config.vocab_size:,}")
    print(f"Maximum context length: {model.config.n_positions:,}")
    print(f"Total parameters      : {total_parameters:,}")
    print(f"Trainable parameters  : {trainable_parameters:,}")
    print("=" * 55 + "\n")
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
    
        # GPT-2 computes cross-entropy loss automatically when labels are passed
        outputs = model(input_ids=input_ids, labels=labels)
        
        loss = outputs.loss
        logits = outputs.logits
    
        print(f"Initial Loss: {loss.item():.4f}")
        print(f"Logits Shape: {logits.shape}")  # [batch_size, sequence_length, vocab_size]
        break

@torch.inference_mode()
def generate_responses(
    model,
    tokenizer,
    prompts,
    max_new_tokens=50,
    batch_size=4):
    """
    Generate deterministic completions and decode only the model response,
    excluding the original prompt.
    """
    if not prompts:
        return []

    model.eval()
    device = next(model.parameters()).device

    # GPT-2 does not define a padding token by default.
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Left padding is preferred for batched decoder-only generation.
    tokenizer.padding_side = "left"

    model_context_length = getattr(
        model.config,
        "n_positions",
        tokenizer.model_max_length,
    )

    max_prompt_length = model_context_length - max_new_tokens
    if max_prompt_length <= 0:
        raise ValueError(
            "max_new_tokens must be smaller than the model context length"
        )

    responses = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start : start + batch_size]

        encoded_inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_prompt_length,
            return_attention_mask=True,
        )

        encoded_inputs = {
            key: value.to(device)
            for key, value in encoded_inputs.items()
        }

        generated_ids = model.generate(
            **encoded_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        # All input rows have the same padded width. Everything after this
        # position is newly generated model output.
        prompt_width = encoded_inputs["input_ids"].shape[1]
        response_token_ids = generated_ids[:, prompt_width:]

        batch_responses = tokenizer.batch_decode(
            response_token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        responses.extend(response.strip() for response in batch_responses)

    return responses

In [35]:
model_dict = load_gpt2_model()
check_model(dataloader, model_dict["model"], model_dict["device"])


Using device: cuda
Successfully loaded gpt2 with 124,439,808 parameters.

                 MODEL INFORMATION
Model name/path       : gpt2
Model architecture    : gpt2
Transformer layers    : 12
Attention heads       : 12
Embedding dimension   : 768
Vocabulary size       : 50,257
Maximum context length: 1,024
Total parameters      : 124,439,808
Trainable parameters  : 124,439,808



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Initial Loss: 2.8303
Logits Shape: torch.Size([8, 1024, 50257])


In [28]:
import json
import torch
from pathlib import Path

def capture_baseline_generations(
    model, 
    tokenizer, 
    prompts: list[str], 
    device: torch.device,
    output_file: str = "baseline_outputs.json",
    max_new_tokens: int = 100,
    seed: int = 42
):
    """
    Generates text for a list of prompts and saves the outputs with generation settings.
    """
    # Set seed for reproducible baseline comparison
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model.eval()
    baseline_results = {
        "metadata": {
            "model_name": model.config._name_or_path,
            "max_new_tokens": max_new_tokens,
            "seed": seed,
            "temperature": 0.7,
            "top_p": 0.9,
        },
        "samples": []
    }

    print("\n--- Capturing Baseline Outputs ---\n")

    with torch.no_grad():
        for idx, prompt in enumerate(prompts):
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            
            # Decode full text and generated-only text
            full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
            generated_only = tokenizer.decode(
                output_tokens[0][inputs.input_ids.shape[1]:], 
                skip_special_tokens=True
            )

            sample_entry = {
                "id": idx + 1,
                "prompt": prompt,
                "generated_continuation": generated_only,
                "full_text": full_text
            }
            
            baseline_results["samples"].append(sample_entry)
            
            print(f"Prompt [{idx + 1}/{len(prompts)}]: {prompt}")
            print(f"Baseline Output:\n{generated_only}\n")
            print("-" * 50)

    # Save to disk
    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(baseline_results, f, indent=2, ensure_ascii=False)

    print(f"Baseline outputs saved to: {out_path.resolve()}\n")
    return baseline_results

In [31]:
# Define test prompts
baseline_prompts = [
    # Domain-specific prompts (matching your CPT corpus)
    "The switches from cisco for data center",
    "BGP protocol is responsible for",
    # General language completion prompts
    "The best data center switch is ",
    "Water freezes at a temperature of",
    "The largest planet in the Solar System is"
]

# Run capture
baseline_data = capture_baseline_generations(
    model=model_dict["model"],
    tokenizer=model_dict["tokenizer"],
    prompts=baseline_prompts,
    device=model_dict["device"],
    output_file="baselines/gpt2_pre_cpt_baseline.json",
    max_new_tokens=80,
    seed=42
)


--- Capturing Baseline Outputs ---

Prompt [1/5]: The switches from cisco for data center
Baseline Output:
 to the Internet of Things (IoT) have become so ubiquitous that it's not surprising that there are a number of companies that have dedicated their entire careers to building these new technologies.

IoT is the technology behind the Internet of Things, and it's the one that has been most successful. In fact, as recently as the mid-1990s, Cisco was the world's

--------------------------------------------------
Prompt [2/5]: BGP protocol is responsible for
Baseline Output:
 the development of the internet's first "full-fledged" public-key cryptography, which is a fundamental part of the cryptographic infrastructure that enables all kinds of secure communication.

As the world's most popular digital currency, bitcoin has been a global success story, and the cryptocurrency's market capitalization is now estimated at $25 billion, making it the world's most widely used digital currency

In [37]:
# Load baseline prompts from JSON

import json
from datetime import datetime, timezone
from pathlib import Path

import torch


def load_prompt_json(json_path):
    """
    Load prompt records and create a simple prompt array.

    Returns:
        records: Complete query records, including IDs and expected concepts.
        prompts: List of prompt strings for model generation.
    """
    json_path = Path(json_path)

    with json_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    queries = data.get("queries")
    if not isinstance(queries, list):
        raise ValueError(f"'queries' must be a list in {json_path}")

    records = []
    prompts = []

    for index, query in enumerate(queries):
        prompt = query.get("prompt")

        if not isinstance(prompt, str) or not prompt.strip():
            raise ValueError(
                f"Query at index {index} does not contain a valid prompt"
            )

        records.append(query)
        prompts.append(prompt.strip())

    print(f"Loaded {len(prompts)} prompts from: {json_path}")

    return records, prompts

def create_result_records(query_records, responses):
    """
    Combine each query with its generated response and evaluation metadata.
    """
    if len(query_records) != len(responses):
        raise ValueError(
            "The number of query records and responses must match"
        )

    results = []

    for query, response in zip(query_records, responses):
        results.append(
            {
                "id": query.get("id"),
                "category": query.get("category"),
                "prompt": query.get("prompt"),
                "response": response,
                "expected_concepts": query.get(
                    "expected_concepts",
                    [],
                ),
            }
        )

    return results

def save_generation_results(
    results,
    output_path,
    model,
    max_new_tokens,
    evaluation_stage="pre_cpt",
):
    """
    Save generated responses and generation settings as formatted JSON.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_data = {
        "evaluation_stage": evaluation_stage,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_name": model.config._name_or_path,
        "generation_configuration": {
            "max_new_tokens": max_new_tokens,
            "do_sample": False,
            "decoding": "greedy",
        },
        "total_prompts": len(results),
        "results": results,
    }

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(
            output_data,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Saved {len(results)} responses to: {output_path}")



In [39]:
domain_records, domain_prompts = load_prompt_json("baselines/domain_baseline_queries.json")
forget_records, forget_prompts = load_prompt_json("baselines/generic_forgetting_queries.json")


MAX_NEW_TOKENS = 50



responses = generate_responses(
    model=model_dict["model"],
    tokenizer=model_dict["tokenizer"],
    prompts=domain_prompts,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=4,
)

results = create_result_records(
    domain_records,
    responses,
)

save_generation_results(
    results=results,
    output_path="baselines/gpt2_domain_baseline_queries_responses.json",
    model=model,
    max_new_tokens=MAX_NEW_TOKENS,
    evaluation_stage="pre_cpt",
)

responses = generate_responses(
    model=model_dict["model"],
    tokenizer=model_dict["tokenizer"],
    prompts=forget_prompts,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=4,
)

results = create_result_records(
    forget_records,
    responses,
)

save_generation_results(
    results=results,
    output_path="baselines/gpt2_forget_baseline_queries_responses.json",
    model=model,
    max_new_tokens=MAX_NEW_TOKENS,
    evaluation_stage="pre_cpt",
)

Loaded 20 prompts from: baselines/domain_baseline_queries.json
Loaded 15 prompts from: baselines/generic_forgetting_queries.json
Saved 20 responses to: baselines/gpt2_domain_baseline_queries_responses.json
Saved 15 responses to: baselines/gpt2_forget_baseline_queries_responses.json
